In [ ]:
# Building out the Value object, which is just a wrapper around a scalar with sufficient math operations for computing dot products, 
# introducing nonlinearity, and backpropagating gradients through a neural net/computation graph.
import math
class Value:
    # Store scalar data, gradient, and the two children values plus the operation used to build this value. 
    # Allows for smooth backpropagation through a computation graph.
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0

        # backward is used to compute the gradient of this value with respect to its children.
        self._backward = lambda: None # Default: leaf nodes have no gradients to propagate backward.
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(data = self.data + other.data, _children = (self, other), _op = '+')
        def _backward():
            # Ex: out = a + b, L is final result; dL/da = dL/dout * dout/da = dL/dout * 1, dL/db = dL/dout * dout/db = dL/dout * 1. Accumulate gradients
            self.grad += out.grad * 1.0
            other.grad += out.grad * 1.0
        out._backward = _backward
        return out

    def __mul__(self, other): 
        other = other if isinstance(other, Value) else Value(other)
        out = Value(data=self.data * other.data, _children = ( self, other), _op = '*')
        def _backward():
            # Ex: out = a * b, L is final result; dL/da = dL/dout * b, dL/db = dL/dout * a. Accumulate gradients
            self.grad += out.grad * other.data 
            other.grad += out.grad * self.data 
        out._backward = _backward
        return out

    # Needed when self is not a Value object, but other is. self * other = other.__rmul__(self)
    def __rmul__(self, other):
        self = self if isinstance(self, Value) else Value(self)
        return self.__mul__(other)

    def __truediv__(self, other):
        # Ex: a / b, where a and b are Value objects. This calls the __mul__ method with self as the first operand and the reciprocal of other as the second operand.
        other = other if isinstance(other, Value) else Value(other)
        return self.__mul__(other**-1)

    def __rtruediv__(self, other):
        # Ex: a / b, where a and b are Value objects. This calls the __mul__ method with self as the first operand and the reciprocal of other as the second operand.
        self = self if isinstance(self, Value) else Value(self)
        return self.__mul__(self**-1)

    def exp(self):
        x = self.data
        out = Value(data=math.exp(x), _children = (self), _op = 'exp')
        def _backward():
            # Ex: out = exp(a), L is final result; dL/da = dL/dout * dout/da = dL/dout * exp(a). Accumulate gradients
            self.grad += out.grad * out.data 
        out._backward = _backward
        return out

    # Hypoerbolic tangent activation function
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(data=t, _children = (self), _op = 'tanh')

        def _backward():
            # Ex: out = tanh(a), L is final result; dL/da = dL/dout * dout/da = dL/dout * (1 - tanh(a)^2). Accumulate gradients
            self.grad += out.grad * (1 - t**2)   
        out._backward = _backward
        return out

    # Backward pass to compute gradients of all values in the computation graph with respect to the final output value.
    def backward(self):
        # Topological sort of the computation graph to ensure we compute gradients in the correct order.
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # Initialize the gradient of the final output value to 1.0 (dL/dL = 1).
        self.grad = 1.0
        # Traverse the graph in reverse topological order and call the backward function for each value.
        for v in reversed(topo):
            v._backward()
        

Value(data=5)